# TrackViewer Visualization of Filtered STRs and Functional Annotations

This notebook uses the [trackViewer](https://bioconductor.org/packages/trackViewer) (Bioconductor) R package to visualize the **8 filtered STRs** from `Relatorio_STR_Final_Integral.pdf` together with their functional annotations.

**Inputs**
- Local project data: STR coordinates / residual / scRNA-seq expression
- External tracks downloaded by `7.4.3.1_download_external_tracks.sh` into `external_tracks/`

**Reference genome:** hg38 (GRCh38).

In [34]:
suppressPackageStartupMessages({
  library(trackViewer)
  library(GenomicRanges)
  library(rtracklayer)
  library(TxDb.Hsapiens.UCSC.hg38.knownGene)
  library(readr)
})

cat('trackViewer loaded OK\n')

trackViewer loaded OK


## 2. Define the 8 filtered STRs

The 8 STR loci are loaded from the unified scRNA-seq overlap file generated in step 7.3.1
(`../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv`), taking one row per locus.
Genomic coordinates (`start0` 0-based, `end`) are taken from `STR_variants_UCSC_track.bed`
(shipped with this notebook), falling back to `start0 + nchar(motif) * copy` if the BED is absent.


In [35]:
# 2. Define the 8 filtered STRs from the unified overlap CSV + BED coordinates
variants_file <- '../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv'
bed_file <- 'STR_variants_UCSC_track.bed'

if (file.exists(variants_file)) {
  # one row per STR locus (drop duplicated cell_type rows)
  ov <- read_csv(variants_file, show_col_types = FALSE)
  variants <- ov[!duplicated(ov$STRs_ID), ]

  # STRs_ID encodes chr:pos:motif:copy  (pos = 1-based start)
  id_parts <- strsplit(variants$STRs_ID, ':', fixed = TRUE)
  variants$chr    <- vapply(id_parts, function(x) x[1], character(1))
  variants$start1 <- as.integer(vapply(id_parts, function(x) x[2], character(1)))
  variants$motif  <- vapply(id_parts, function(x) x[3], character(1))
  variants$copy   <- as.integer(vapply(id_parts, function(x) x[4], character(1)))
  variants$start0 <- variants$start1 - 1
  variants$gene   <- variants$gene_name
  variants$allele2 <- variants$allele2_est
  variants$group  <- ifelse(tolower(variants$group) == 'case', 'Case', 'Control')

  # BED coordinates for start0 / end (0-based start, 1-based-style end)
  if (file.exists(bed_file)) {
    bed <- read.delim(bed_file, comment.char = '#', header = FALSE, fill = TRUE,
                      col.names = c('chr', 'start0', 'end', 'name'))
    bed <- bed[grepl('^chr', bed$chr), ]
    bed$STRs_ID <- sub('^[^_]+_', '', bed$name)  # name = <gene>_<STRs_ID>
    bed <- bed[bed$STRs_ID %in% variants$STRs_ID, c('STRs_ID', 'end')]
    variants$end <- bed$end[match(variants$STRs_ID, bed$STRs_ID)]
    if (anyNA(variants$end)) {
      warning('Some loci missing from BED; estimating end from motif length')
      na <- is.na(variants$end)
      variants$end[na] <- variants$start0[na] + nchar(variants$motif[na]) * variants$copy[na]
    }
  } else {
    cat('BED file not found; estimating end from motif length * copy\n')
    variants$end <- variants$start0 + nchar(variants$motif) * variants$copy
  }

  cat('Loaded', nrow(variants), 'STR loci from', variants_file, '\n')
} else {
  stop('Overlap CSV not found: ', variants_file)
}

print(variants[, c('STRs_ID', 'gene', 'abs_res', 'allele2', 'group', 'chr', 'start0', 'end')])

# GRanges of the STRs (1-based start)
gr_strs <- GRanges(seqnames = variants$chr,
                   ranges = IRanges(start = variants$start0 + 1, end = variants$end),
                   strand = '*',
                   STRs_ID = variants$STRs_ID,
                   gene = variants$gene,
                   abs_res = variants$abs_res,
                   allele2 = variants$allele2,
                   group = variants$group,
                   motif = variants$motif)
names(gr_strs) <- variants$gene
gr_strs

Loaded 8 STR loci from ../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv 


## 3. Load external tracks

Files are expected in `external_tracks/` (created by the download script). Each track is a helper function that returns a `trackViewer` feature track (importing bigWig/BED via `rtracklayer::import`).

In [ ]:
ext_dir <- 'external_tracks'
stopifnot(dir.exists(ext_dir))

win_all <- if (exists('gr_strs') && length(gr_strs) > 0) {
  reduce(resize(gr_strs, width = 20000, fix = 'center'))
} else {
  if (exists('variants') && nrow(variants) > 0) {
    gr_fallback <- GRanges(
      seqnames = variants$chr,
      ranges = IRanges(start = variants$start0 + 1, end = variants$end),
      strand = '*'
    )
    reduce(resize(gr_fallback, width = 20000, fix = 'center'))
  } else {
    NULL
  }
}

is_bigwig <- function(path) {
  con <- file(path, 'rb'); on.exit(close(con))
  magic <- readBin(con, 'raw', n = 4)
  length(magic) == 4 && (
    identical(magic, as.raw(c(0x26, 0xfc, 0x8f, 0x88))) ||
    identical(magic, as.raw(c(0x88, 0x8f, 0xfc, 0x26)))
  )
}

is_text_error <- function(path) {
  con <- file(path, 'rb'); on.exit(close(con))
  magic <- readBin(con, 'raw', n = 2)
  if (length(magic) == 2 && identical(magic, as.raw(c(0x1f, 0x8b)))) return(FALSE)
  head <- readChar(con, nchars = 200, useBytes = TRUE)
  grepl('<html|<!DOCTYPE', head, ignore.case = TRUE, useBytes = TRUE)
}

import_bw <- function(path, name, color, win = NULL) {
  if (!file.exists(path)) {
    warning('Missing file: ', path)
    return(NULL)
  }
  if (!is_bigwig(path)) {
    warning('Not a valid bigWig (magic bytes), skipping: ', path)
    return(NULL)
  }
  gr <- rtracklayer::import(path, which = win)
  if (length(gr) == 0) return(NULL)
  if (is.null(gr$score)) mcols(gr)$score <- 0
  tr <- new("track", dat = gr, type = "data", format = "BigWig", name = name)
  setTrackStyleParam(tr, "color", color)
  tr
}

import_bed <- function(path, name, color, win = NULL) {
  if (!file.exists(path)) {
    warning('Missing file: ', path)
    return(NULL)
  }
  if (is_text_error(path)) {
    warning('Looks like an error/HTML page, skipping: ', path)
    return(NULL)
  }
  gr <- rtracklayer::import(path, format = 'BED', which = win)
  if (length(gr) == 0) return(NULL)
  if (is.null(gr$score)) mcols(gr)$score <- 1
  tr <- new("track", dat = gr, type = "data", format = "BED", name = name)
  setTrackStyleParam(tr, "color", color)
  tr
}

bigbed_bin <- Sys.which('bigBedToBed')
if (!nzchar(bigbed_bin)) {
  cand <- file.path(Sys.getenv('MAMBA_ROOT_PREFIX', '/home/matheus/micromamba'),
                    'envs/ucsc/bin/bigBedToBed')
  if (file.exists(cand)) bigbed_bin <- cand
}

import_bigbed <- function(path, name, color, win = NULL) {
  if (!nzchar(bigbed_bin) || !file.exists(bigbed_bin)) {
    warning('bigBedToBed not found; skipping bigBed track: ', name)
    return(NULL)
  }
  if (!file.exists(path)) {
    warning('Missing file: ', path)
    return(NULL)
  }
  if (is.null(win) || length(win) == 0) return(NULL)
  gr_list <- lapply(seq_along(win), function(k) {
    w <- win[k]
    out_bed <- tempfile(fileext = '.bed')
    ok <- tryCatch(
      system2(bigbed_bin, args = c('-chrom', as.character(seqnames(w)),
                                   '-start', as.character(start(w) - 1),
                                   '-end', as.character(end(w)),
                                   path, out_bed)),
      error = function(e) 1L)
    if (ok != 0 || !file.exists(out_bed) || file.size(out_bed) == 0) {
      unlink(out_bed)
      return(GRanges())
    }
    gr <- tryCatch(rtracklayer::import(out_bed, format = 'BED'),
                   error = function(e) GRanges())
    unlink(out_bed)
    gr
  })
  gr <- do.call(c, gr_list)
  if (length(gr) == 0) return(NULL)
  if (is.null(gr$score)) mcols(gr)$score <- 1
  tr <- new("track", dat = gr, type = "data", format = "BED", name = name)
  setTrackStyleParam(tr, "color", color)
  tr
}

tracks_external <- list(
  CTCF     = import_bw(file.path(ext_dir, 'CTCF_ENCFF910VLV.bw'),       'CTCF ChIP-seq',         '#D7301F', win_all),
  DNase    = import_bw(file.path(ext_dir, 'DNase_brain.bw'),            'DNase',                  '#E08214', win_all),
  H3K27ac  = import_bw(file.path(ext_dir, 'H3K27ac_brain.bw'),          'H3K27ac',                '#8073AC', win_all),
  H3K27me3 = import_bw(file.path(ext_dir, 'H3K27me3_brain.bw'),         'H3K27me3',               '#4E9A06', win_all),
  RemapD   = import_bw(file.path(ext_dir, 'remap2022_density_hg38.bw'), 'ReMap Density',          '#4575B4', win_all),
  ReMapAtlas = import_bw(file.path(ext_dir, 'remap_density_ucsc.bw'),  'ReMap Atlas',            '#2C7BB6', win_all),
  UKBdeCODE = import_bw(file.path(ext_dir, 'ukbDepletion.bw'),         'UKB/deCODE Depletion Rank', '#984EA3', win_all),
  wgEncode  = import_bw(file.path(ext_dir, 'wgEncodeReg4TfChip_ENCFF341CQE.bw'), 'CTCF wgEncodeReg4TfChip', '#A65628', win_all)
)

tracks_external$JARVIS <- NULL
if (file.exists(file.path(ext_dir, 'jarvis.bw'))) {
  tracks_external$JARVIS <- import_bw(file.path(ext_dir, 'jarvis.bw'),
                                       'JARVIS', '#6A3D9A', win_all)
} else {
  jarvis_url <- 'https://hgdownload.soe.ucsc.edu/gbdb/hg38/jarvis/jarvis.bw'
  grj <- tryCatch(rtracklayer::import(rtracklayer::BigWigFile(jarvis_url), which = win_all),
                  error = function(e) NULL)
  if (!is.null(grj) && length(grj) > 0) {
    trj <- new("track", dat = grj, type = "data", format = "BigWig", name = 'JARVIS')
    setTrackStyleParam(trj, "color", '#6A3D9A')
    tracks_external$JARVIS <- trj
  } else {
    warning('JARVIS track unavailable (remote query failed and no local jarvis.bw)')
  }
}

tracks_external$JASPAR    <- import_bigbed(file.path(ext_dir, 'JASPAR2024.bb'),         'JASPAR CORE TFBS',  '#377EB8', win_all)
tracks_external$DECIPHER  <- import_bigbed(file.path(ext_dir, 'decipher_common.bb'),    'DECIPHER common CNV', '#E41A1C', win_all)
tracks_external$TRExplorer <- import_bigbed(file.path(ext_dir, 'trexplorer.bb'),        'TRExplorer',        '#4DAF4A', win_all)

rmsk_path <- file.path(ext_dir, 'rmsk.hg38.bed.gz')
if (file.exists(rmsk_path)) {
  tracks_external$RepeatMasker <- import_bed(rmsk_path, 'RepeatMasker hg38', '#8C510A', win_all)
  gr_rmsk <- tryCatch(rtracklayer::import(rmsk_path, format = 'BED', which = win_all),
                      error = function(e) GRanges())
  if (length(gr_rmsk) > 0 && !is.null(gr_rmsk$name)) {
    dfam <- gr_rmsk[grepl('^(LINE|L1|ERV|DNA)', gr_rmsk$name), ]
    if (length(dfam) > 0) {
      trd <- new("track", dat = dfam, type = "data", format = "BED", name = 'DFam')
      setTrackStyleParam(trd, "color", '#A6611A')
      tracks_external$DFam <- trd
    }
  }
} else {
  warning('RepeatMasker file not found: ', rmsk_path)
}

gtex_path <- file.path(ext_dir, 'gtex_eqTL_rsids.tsv')
if (file.exists(gtex_path)) {
  gtex <- read.delim(gtex_path, header = TRUE, stringsAsFactors = FALSE)
  if (nrow(gtex) > 0 && file.exists(file.path(ext_dir, 'gtex_eqtl.bed'))) {
    tracks_external$GTEx <- import_bed(file.path(ext_dir, 'gtex_eqtl.bed'),
                                        'GTEx cis-eQTL', '#F781BF', win_all)
  } else {
    cat('GTEx rsIDs present but gtex_eqtl.bed not found; skipping GTEx track\n')
  }
}

tf_names <- c('CTCF','REST','KLF9','GATA2','ZNF384','MNT',
              'NACC2','FOXB1','FOXK1','PATZ1','RAD21','TCF4','MITF','USF2','RAD51','GATA1')
tf_files <- c(
  'remap2022_ctcf_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_rest_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_klf9_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_gata2_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_znf384_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_mnt_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_nacc2_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_foxb1_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_foxk1_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_patz1_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_rad21_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_tcf4_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_mitf_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_usf2_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_rad51_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_gata1_all_macs2_hg38_v1_0.bed.gz'
)
for (i in seq_along(tf_names)) {
  tr <- import_bed(file.path(ext_dir, tf_files[i]), tf_names[i], '#238B45', win_all)
  if (!is.null(tr)) tracks_external[[paste0(tf_names[i], '_peaks')]] <- tr
}

cat('External tracks loaded:', sum(!vapply(tracks_external, is.null, logical(1))), '/', length(tracks_external), '\n')

## 4. Load local project data (scRNA-seq expression)

Optional overlay: LogFC per cell type from the unified overlap CSV (`../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv`), shipped in this repo, used to color/adjust the STR features.


In [37]:
scrna_file <- '../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv'
if (file.exists(scrna_file)) {
  scrna <- read_csv(scrna_file, show_col_types = FALSE)
  scrna <- scrna[!is.na(scrna$LogFC) & !is.na(scrna$STRs_ID), ]
  cat('scRNA overlap rows:', nrow(scrna), '\n')
  print(unique(scrna[, c('gene_name','STRs_ID','source_tissue','LogFC')]))
} else {
  cat('scRNA overlap file not found; skipping overlay\n')
  scrna <- NULL
}

scRNA overlap rows: 15 


## 5. Gene model track

Build a gene track per locus from the TxDb (hg38 knownGene).

In [38]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene

gene_track_for <- function(gr_variant) {
  win <- gr_variant
  start(win) <- start(win) - 20000
  end(win)   <- end(win)   + 20000
  
  gt <- geneModelFromTxdb(txdb, gr = win)
  
  if (length(gt) > 1) {
    gt <- gt[1]
  }
  
  return(gt)
}

cat('Gene model helper ready\n')

Gene model helper ready


## 6. Variant track with annotations

Each STR is drawn as a feature with height scaled by absolute residual and colored by group.

In [46]:
make_str_track <- function(gr_variant) {
  center_pos <- start(gr_variant) + floor(width(gr_variant) / 2)
  pos_seq <- seq(center_pos - 250, center_pos + 250, by = 10)
  
  gr <- GRanges(
    seqnames = seqnames(gr_variant),
    ranges = IRanges(start = pos_seq, width = 10),
    strand = "*"
  )
  mcols(gr)$score <- 1
  
  track_label <- paste0("STR: ", gr_variant$gene)
  tr <- new("track", dat = gr, type = "data", format = "BED", name = track_label)
  
  col_group <- ifelse(gr_variant$group == 'Case', '#C62828', '#1565C0')
  setTrackStyleParam(tr, "color", col_group)
  setTrackStyleParam(tr, "height", 0.08)
  
  return(tr)
}

make_scrna_track <- function(scrna, gr_variant) {
  if (is.null(scrna)) return(NULL)
  sub <- scrna[scrna$STRs_ID == gr_variant$STRs_ID, ]
  if (nrow(sub) == 0) return(NULL)
  
  gr <- resize(gr_variant, width = 300, fix = "center")
  mcols(gr)$score <- mean(sub$LogFC, na.rm = TRUE)
  tr <- new("track", dat = gr, type = "data", format = "BED",
            name = paste0(gr_variant$gene, ' LogFC'))
  setTrackStyleParam(tr, "color", '#6A51A3')
  setTrackStyleParam(tr, "height", 0.08)
  tr
}

In [ ]:
dir.create('results', showWarnings = FALSE)

make_str_track <- function(gr_variant, clean_id) {
  center_pos <- start(gr_variant) + floor(width(gr_variant) / 2)
  pos_seq <- seq(center_pos - 100, center_pos + 100, by = 10)
  
  gr <- GRanges(
    seqnames = seqnames(gr_variant),
    ranges = IRanges(start = pos_seq, width = 10),
    strand = "*"
  )
  mcols(gr)$score <- 1
  
  tr <- new("track", dat = gr, type = "data", format = "BED", name = clean_id)
  
  col_group <- ifelse(gr_variant$group == 'Case', '#C62828', '#1565C0')
  setTrackStyleParam(tr, "color", col_group)
  setTrackStyleParam(tr, "height", 0.08)
  setTrackYaxisParam(tr, "draw", FALSE)
  
  return(tr)
}

render_variant <- function(view_window, track_list, out_png) {
  png(out_png, width = 1400, height = 900, res = 150)
  
  viewer_style <- trackViewerStyle()
  viewer_style@margin <- c(0.05, 0.18, 0.05, 0.02)
  
  viewTracks(track_list, 
             gr = view_window,
             viewerStyle = viewer_style,
             autoOptimizeStyle = TRUE)
             
  dev.off()
  cat('saved:', out_png, '\n')
}

variant_tracks <- list(
  'chr1:211045041:AT:9'  = c('JARVIS', 'UKBdeCODE', 'GTEx', 'RepeatMasker',
                              'DECIPHER', 'TRExplorer', 'wgEncode'),
  'chr1:76143392:GT:16'  = c('JARVIS', 'UKBdeCODE', 'DNase', 'H3K27ac',
                              'RepeatMasker', 'TRExplorer', 'DECIPHER',
                              'RemapD', 'KLF9_peaks', 'NACC2_peaks'),
  'chr3:76185195:AT:11'  = c('JARVIS', 'UKBdeCODE')
)